In [ ]:
# Install Geopandas jika di jalankan di environment baru (Colab biasanya sudah ada)
# !pip install geopandas shapely matplotlib

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

print("=== TAHAP 1: DATA GENERATION & CRS TRANSFORMATION ===")
# 1. GENERATE FASILITAS PUBLIK (Klinik Eksisting di Bengkulu)
lokasi_klinik = {
    'nama': ['Klinik Pusat', 'Klinik Utara', 'Klinik Selatan', 'Klinik Timur'],
    'geometry': [
        Point(102.2600, -3.8000), 
        Point(102.2750, -3.7800), 
        Point(102.2550, -3.8250),
        Point(102.2850, -3.8150)
    ]
}
gdf_klinik = gpd.GeoDataFrame(lokasi_klinik, crs="EPSG:4326")

# 2. GENERATE PERMUKIMAN WARGA (300 Titik Sintetis acak di sekitar kota)
np.random.seed(42)
n_warga = 300
lon_warga = np.random.uniform(102.2300, 102.3100, n_warga) # Bounding box Bujur
lat_warga = np.random.uniform(-3.8500, -3.7600, n_warga)   # Bounding box Lintang

lokasi_warga = [Point(lon, lat) for lon, lat in zip(lon_warga, lat_warga)]
gdf_warga = gpd.GeoDataFrame({'id_permukiman': range(1, n_warga+1)}, geometry=lokasi_warga, crs="EPSG:4326")

print(f"Total Fasilitas : {len(gdf_klinik)} Titik")
print(f"Total Permukiman: {len(gdf_warga)} Titik")

# 3. TRANSFORMASI CRS (EPSG:4326/Derajat Bumi -> EPSG:32748/Meter UTM Zone 48S)
# Ini wajib dilakukan agar perhitungan radius jarak 100% akurat dalam satuan meter
gdf_klinik_meter = gdf_klinik.to_crs(epsg=32748)
gdf_warga_meter = gdf_warga.to_crs(epsg=32748)


print("\n=== TAHAP 2: GEOPROCESSING & SPATIAL INTERSECTION ===")
# 1. Buffering: Membuat radius pelayanan sejauh 2 Kilometer (2000 meter)
print("-> Membuat poligon radius pelayanan sejauh 2 KM...")
gdf_klinik_meter['buffer_2km'] = gdf_klinik_meter.geometry.buffer(2000)

# Menggabungkan semua radius menjadi satu kesatuan poligon cakupan (Dissolve)
cakupan_gabungan = gdf_klinik_meter['buffer_2km'].unary_union

# 2. Spatial Intersection: Memisahkan wilayah terjangkau vs Blind Spot
print("-> Mengeksekusi analisis topologi irisan (Intersect)...")
gdf_warga_meter['status'] = gdf_warga_meter.geometry.apply(
    lambda x: 'Terjangkau' if cakupan_gabungan.contains(x) else 'Blind Spot'
)

# Menghitung Metrik Bisnis/Layanan
total_terjangkau = len(gdf_warga_meter[gdf_warga_meter['status'] == 'Terjangkau'])
total_blindspot = len(gdf_warga_meter[gdf_warga_meter['status'] == 'Blind Spot'])
coverage_ratio = (total_terjangkau / n_warga) * 100

print("\n--- LAPORAN ANALITIK TATA RUANG ---")
print(f"Total Permukiman Aman       : {total_terjangkau} titik")
print(f"Total Permukiman BLIND SPOT : {total_blindspot} titik")
print(f"Rasio Cakupan Kota          : {coverage_ratio:.2f}%")
print("Rekomendasi                 : Fokuskan anggaran pembangunan di area klaster Blind Spot.")


print("\n=== TAHAP 3: DATA VISUALIZATION ===")
# Mengembalikan CRS ke bentuk awal (Derajat) untuk keperluan visualisasi peta
gdf_klinik_buffer_vis = gdf_klinik_meter.set_geometry('buffer_2km').to_crs(epsg=4326)
gdf_warga_vis = gdf_warga_meter.to_crs(epsg=4326)

# Memisahkan data untuk pewarnaan plot
warga_aman = gdf_warga_vis[gdf_warga_vis['status'] == 'Terjangkau']
warga_bahaya = gdf_warga_vis[gdf_warga_vis['status'] == 'Blind Spot']

# Mulai menggambar Peta
fig, ax = plt.subplots(figsize=(10, 8))

# Plot 1: Poligon Radius Layanan (Warna Biru Transparan)
gdf_klinik_buffer_vis.plot(ax=ax, color='blue', alpha=0.15, edgecolor='blue', linewidth=1.5, label='Radius Layanan (2 KM)')

# Plot 2: Titik Fasilitas Klinik (Bintang Biru)
gdf_klinik.plot(ax=ax, marker='*', color='blue', markersize=250, zorder=5, label='Klinik / Fasilitas')

# Plot 3: Titik Warga Terjangkau (Titik Hijau)
warga_aman.plot(ax=ax, marker='o', color='#2ecc71', markersize=30, alpha=0.8, label='Permukiman Terjangkau')

# Plot 4: Titik Warga Blind Spot (Titik Merah X)
warga_bahaya.plot(ax=ax, marker='X', color='#e74c3c', markersize=40, alpha=0.9, label='Permukiman BLIND SPOT')

# Mempercantik Visualisasi
ax.set_title('Dashboard Analitik Spasial: Deteksi Blind Spot Fasilitas Publik\n(Kota Bengkulu)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Garis Bujur (Longitude)')
ax.set_ylabel('Garis Lintang (Latitude)')
ax.legend(loc='upper right', frameon=True, shadow=True)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()